# B09 — Round-1 full eval, both types (`/predict`)

Run the live `/predict` endpoint over **every** round-1 test case in
`outputs/exact_eval_round1_Prompt2Win.json` (25 type1 + 25 type2) and score it
the way the organizer does.

**Scoring (per the round-1 result file):**
- **type1** (MCQ / Yes-No-Uncertain): `sample = 0.5·P1 + 0.5·P2`. P1 = answer
  correct (matched against gold `answer` + `aliases`). P2 = F1 over `premises_used`.
- **type2** (physics): `sample = P1` only (premises not graded; `p2 = null`).
  P1 = numeric answer match within relative tolerance.

Each request is the stored `request_payload` sent verbatim — the API routes by
`type` itself.

In [1]:
import json, re, time, asyncio
from pathlib import Path
from collections import Counter

import httpx

API_BASE    = "https://api.iamphuckhang.dev"
PREDICT_URL = f"{API_BASE}/predict"

N_SAMPLES   = None        # None = full round-1 set (50)
CONCURRENCY = 3           # low: each request is several LLM calls; avoid timeouts
TIMEOUT     = 60.0
NUM_RTOL    = 0.02        # type2 numeric match: 2% relative tolerance

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "outputs").exists():
    ROOT = ROOT.parent
INPUT_JSON  = ROOT / "outputs/exact_eval_round1_Prompt2Win.json"
OUTPUT_JSON = ROOT / f"outputs/B09_round1_full_{time.strftime('%Y%m%d_%H%M%S')}.json"
print("input   :", INPUT_JSON)
print("output  :", OUTPUT_JSON)
print("endpoint:", PREDICT_URL, "| concurrency:", CONCURRENCY)

input   : /home/phuckhang/MyWorkspace/Exact2026/outputs/exact_eval_round1_Prompt2Win.json
output  : /home/phuckhang/MyWorkspace/Exact2026/outputs/B09_round1_full_20260621_115507.json
endpoint: https://api.iamphuckhang.dev/predict | concurrency: 3


## Load round-1 cases (both types)

In [2]:
logs = json.load(open(INPUT_JSON))["logs"]

samples = []
for log in logs:
    exp = log.get("expected") or {}
    samples.append({
        "query_id": log["query_id"],
        "type": log["type"],
        "category": log.get("category"),
        "payload": log["request_payload"],            # sent verbatim
        "_gold": exp.get("answer"),
        "_aliases": exp.get("aliases") or [],
        "_unit": exp.get("unit") or "",
        "_gold_premises": exp.get("premises_used") or [],
    })

if N_SAMPLES:
    samples = samples[:N_SAMPLES]
all_samples = samples
print(f"loaded {len(all_samples)} cases | by type: {dict(Counter(s['type'] for s in all_samples))}"
      f" | by category: {dict(Counter(s['category'] for s in all_samples))}")

loaded 50 cases | by type: {'type1': 25, 'type2': 25} | by category: {'mcq': 13, 'yes_no_uncertain': 12, 'physics': 25}


## Send to `/predict`

In [3]:
async def call(client, sem, sample):
    async with sem:
        t0 = time.perf_counter()
        err, body = None, None
        try:
            r = await client.post(PREDICT_URL, json=sample["payload"], timeout=TIMEOUT)
            r.raise_for_status()
            body = r.json()
        except Exception as e:
            err = repr(e)
    return {**sample, "_response": body, "_latency": time.perf_counter() - t0, "_error": err}

async def run_eval(samples):
    sem = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(s):
            nonlocal done
            res = await call(client, sem, s)
            done += 1
            if done % 5 == 0 or done == len(samples):
                print(f"  {done}/{len(samples)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(s) for s in samples))

print(f"Sending {len(all_samples)} requests (concurrency={CONCURRENCY})...")
t0 = time.perf_counter()
results = await run_eval(all_samples)
wall = time.perf_counter() - t0
success = [r for r in results if not r["_error"]]
print(f"\nSuccess: {len(success)}/{len(results)}  |  errors: {len(results) - len(success)}  |  wall: {wall:.1f}s")

Sending 50 requests (concurrency=3)...
  50/50
Success: 50/50  |  errors: 0  |  wall: 176.7s


## Score (type-aware)

In [4]:
_UNC = {"uncertain", "unknown"}
# Metric prefixes for unit-aware type2 matching (gold "150 uF" == pred "1.5e-4 F").
_PREFIX = {"T": 1e12, "G": 1e9, "M": 1e6, "k": 1e3,
           "m": 1e-3, "u": 1e-6, "μ": 1e-6, "n": 1e-9, "p": 1e-12}

def _norm(a) -> str:
    s = str(a).strip().lower()
    return "uncertain" if s in _UNC else s

def parse_num(s):
    """Parse a physics answer string to float: '3.38 × 10^-3', '1.2e3', '12 V'."""
    if s is None:
        return None
    t = str(s).strip().replace(",", "").replace("−", "-").replace("×", "x")
    m = re.search(r"([-+]?\d*\.?\d+)\s*[x*]\s*10\s*\^?\s*([-+]?\d+)", t)
    if m:
        return float(m.group(1)) * 10 ** int(m.group(2))
    m = re.search(r"([-+]?\d*\.?\d+)\s*[eE]\s*([-+]?\d+)", t)
    if m:
        return float(f"{m.group(1)}e{m.group(2)}")
    m = re.search(r"[-+]?\d*\.?\d+", t)
    return float(m.group(0)) if m else None

def unit_factor(unit) -> float:
    """Metric-prefix factor of a unit (uF -> 1e-6, kV -> 1e3, F/V/J -> 1)."""
    u = (unit or "").strip()
    return _PREFIX[u[0]] if len(u) >= 2 and u[0] in _PREFIX else 1.0

def _si(val, unit):
    v = parse_num(val)
    return None if v is None else v * unit_factor(unit)

def answer_ok(sample, pred_ans, pred_unit) -> bool:
    if pred_ans is None:
        return False
    if sample["type"] == "type2":
        # compare in SI (value × unit-prefix), 2% relative tolerance
        g = _si(sample["_gold"], sample["_unit"])
        p = _si(pred_ans, pred_unit)
        if g is None or p is None:
            return False
        return abs(p) < 1e-12 if g == 0 else abs(g - p) / abs(g) <= NUM_RTOL
    gold_set = {_norm(sample["_gold"])} | {_norm(a) for a in sample["_aliases"]}
    return _norm(pred_ans) in gold_set

def _f1(gold, pred) -> float:
    g, p = set(gold), set(pred)
    if not g and not p:
        return 100.0
    if not g or not p:
        return 0.0
    inter = len(g & p)
    return 100.0 * 2 * inter / (len(g) + len(p)) if inter else 0.0

scored = []
for r in results:
    resp = r["_response"][0] if r["_response"] else None
    pred_ans = resp.get("answer") if resp else None
    pred_unit = resp.get("unit") if resp else None
    pred_prem = resp.get("premises_used") if resp else None
    ok = answer_ok(r, pred_ans, pred_unit)
    p1 = 100.0 if ok else 0.0
    if r["type"] == "type2":
        p2 = None
        sample_score = p1
    else:
        p2 = _f1(r["_gold_premises"], pred_prem or [])
        sample_score = 0.5 * p1 + 0.5 * p2
    scored.append({
        "query_id": r["query_id"],
        "type": r["type"],
        "category": r["category"],
        "gold_answer": r["_gold"],
        "gold_unit": r["_unit"],
        "pred_answer": pred_ans,
        "pred_unit": pred_unit,
        "answer_ok": ok,
        "gold_premises_used": r["_gold_premises"],
        "pred_premises_used": pred_prem,
        "p1_score": p1,
        "p2_score": (round(p2, 2) if p2 is not None else None),
        "sample_score": round(sample_score, 2),
        "explanation": resp.get("explanation") if resp else None,
        "latency_s": round(r["_latency"], 1),
        "error": r["_error"],
    })

print(f"{'sample':10s} {'type':6s} {'cat':18s} {'gold':14s} {'pred':14s} {'P1':>4s} {'P2':>6s} {'score':>6s}")
print("-" * 88)
for x in scored:
    flag = "✓" if x["answer_ok"] else "✗"
    p2 = "  -  " if x["p2_score"] is None else f"{x['p2_score']:>6.1f}"
    gold = f"{x['gold_answer']}{x['gold_unit']}".strip()
    pred = f"{x['pred_answer']}{x['pred_unit'] or ''}".strip()
    print(f"{x['query_id']:10s} {x['type']:6s} {str(x['category']):18s} "
          f"{gold[:14]:14s} {pred[:14]:14s} "
          f"{x['p1_score']:>4.0f} {p2} {x['sample_score']:>6.1f} {flag}")
    if x["error"]:
        print(f"   !! {x['error']}")

sample     type   cat                gold           pred             P1     P2  score
----------------------------------------------------------------------------------------
T1_0021    type1  mcq                A              A               100  100.0  100.0 ✓
T1_0031    type1  mcq                B              B               100  100.0  100.0 ✓
T1_0025    type1  mcq                C              B                 0   88.9   44.4 ✗
T1_0027    type1  mcq                D              D               100  100.0  100.0 ✓
T1_0035    type1  mcq                D              D               100  100.0  100.0 ✓
T1_0039    type1  mcq                C              C               100  100.0  100.0 ✓
T1_0023    type1  mcq                A              A               100   93.3   96.7 ✓
T1_0033    type1  mcq                C              C               100  100.0  100.0 ✓
T1_0046    type1  mcq                A              A               100  100.0  100.0 ✓
T1_0013    type1  mcq            

## Summary + export

In [5]:
def _avg(xs):
    xs = [x for x in xs if x is not None]
    return round(sum(xs) / len(xs), 2) if xs else 0.0

def _block(rows):
    return {
        "n": len(rows),
        "answer_acc": _avg([100.0 if x["answer_ok"] else 0.0 for x in rows]),
        "p1_avg": _avg([x["p1_score"] for x in rows]),
        "p2_avg": _avg([x["p2_score"] for x in rows]),
        "sample_score_avg": _avg([x["sample_score"] for x in rows]),
    }

by_type = {t: [x for x in scored if x["type"] == t] for t in ("type1", "type2")}
by_cat = {c: [x for x in scored if x["category"] == c]
          for c in sorted({x["category"] for x in scored})}
summary = {
    "overall": _block(scored),
    "by_type": {t: _block(rows) for t, rows in by_type.items() if rows},
    "by_category": {c: _block(rows) for c, rows in by_cat.items()},
    "errors": sum(1 for x in scored if x["error"]),
    "latency_mean_s": _avg([x["latency_s"] for x in scored]),
    "latency_max_s": max((x["latency_s"] for x in scored), default=0.0),
}

payload = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "endpoint": PREDICT_URL,
    "input": str(INPUT_JSON.relative_to(ROOT)),
    "summary": summary,
    "results": scored,
}
OUTPUT_JSON.write_text(json.dumps(payload, indent=2, ensure_ascii=False))

print("=== Round-1 full eval ===")
o = summary["overall"]
print(f"  OVERALL  n={o['n']:<3d} answer_acc={o['answer_acc']:5.1f}%  "
      f"sample={o['sample_score_avg']:5.1f}   <- competition score")
for t, b in summary["by_type"].items():
    extra = f" P2={b['p2_avg']:5.1f}" if t == "type1" else ""
    print(f"  {t:6s} n={b['n']:<3d} answer_acc={b['answer_acc']:5.1f}%  P1={b['p1_avg']:5.1f}{extra}  "
          f"sample={b['sample_score_avg']:5.1f}")
for c, b in summary["by_category"].items():
    print(f"    {c:18s} n={b['n']:<3d} acc={b['answer_acc']:5.1f}%  sample={b['sample_score_avg']:5.1f}")
print(f"  errors={summary['errors']}  latency mean={summary['latency_mean_s']}s  max={summary['latency_max_s']}s")
print(f"\nwrote \u2192 {OUTPUT_JSON}")

=== Round-1 full eval ===
  OVERALL  n=50  answer_acc= 66.0%  sample= 65.9   <- competition score
  type1  n=25  answer_acc= 92.0%  P1= 92.0 P2= 91.7  sample= 91.8
  type2  n=25  answer_acc= 40.0%  P1= 40.0  sample= 40.0
    mcq                n=13  acc= 92.3%  sample= 95.5
    physics            n=25  acc= 40.0%  sample= 40.0
    yes_no_uncertain   n=12  acc= 91.7%  sample= 87.9
  errors=0  latency mean=10.58s  max=33.6s

wrote → /home/phuckhang/MyWorkspace/Exact2026/outputs/B09_round1_full_20260621_115507.json


## Notes
- Input: `outputs/exact_eval_round1_Prompt2Win.json` — the round-1 test (25 type1 + 25 type2). Each case's `request_payload` is POSTed verbatim; the API routes on `type`.
- **type1** scored `0.5·P1 + 0.5·P2` (P1 = answer vs gold+aliases, P2 = F1 over 0-based `premises_used`).
- **type2** scored `P1` only (numeric answer within `NUM_RTOL`; premises not graded, matching the round-1 result file where `p2 = null`).
- `OVERALL.sample_score_avg` = the mean over all 50 = the competition number.
- Lower `CONCURRENCY` if you see `ReadTimeout` (each request is several LLM calls; the GPU saturates under parallel load).
- Inspect one: `next(r for r in results if r['query_id'] == 'T1_0021')['_response']`.